### 16. 로봇 표정 애니메이션 (Vector 스타일)
#### Jetson에 연결된 7인치 디스플레이에 **Anki Vector** 스타일의 로봇 얼굴을 실시간으로 그린다.
##### Vector는 검은 OLED 화면 위에 흰색의 큰 두 눈만으로 감정을 표현한다. 본 실습에서는 matplotlib(TkAgg) 윈도우로 동일한 스타일의 표정(기본/행복/슬픔/화남/놀람/졸림/사랑)을 애니메이션한다.

#### 1) 실행 준비

7인치 화면에 **별도의 창**으로 표시하려면 GUI 백엔드(`TkAgg`)와 `tkinter`가 필요하다. Ubuntu 18.04(JetPack 4.x) 기준:

```bash
sudo apt update
sudo apt install python3-tk
sudo pip3 install numpy matplotlib
```

- 노트북(Jupyter)을 **Jetson의 데스크톱 환경에서 실행**해야 화면에 창이 보인다.
- 디스플레이가 잡혔는지 확인: `xrandr` (7인치 화면이 주 화면으로 보여야 한다)
- 전체 화면(테두리 없는 창) 옵션은 아래 실행 셀의 `FULLSCREEN = True`로 제어한다.

In [ ]:
# 외부 창(7인치 화면) 표시를 위해 TkAgg 백엔드 사용
import matplotlib
matplotlib.use('TkAgg')

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Ellipse, Arc, Polygon
import numpy as np
import math
import time
import random

#### 2) Vector 스타일 얼굴 클래스 정의

- 색상: 검은 배경(`BG_COLOR`), 흰 눈(`EYE_COLOR`), 검은 눈동자(`PUPIL_COLOR`)
- 기본 눈: 둥근 모서리의 흰 사각형 + 안쪽 눈동자(시선 이동 가능)
- 감정별 눈 모양: 행복(∩ 아치+미소), 슬픔(∪ 아치+눈썹), 화남(가늘게 뜬 눈+비스듬한 눈썹), 놀람(크게 뜬 눈), 졸림(반쯤 감긴 눈+Zzz), 사랑(하트 눈)

In [ ]:
# 얼굴 색상 정의
BG_COLOR    = '#000000'   # 얼굴 패널(배경)
EYE_COLOR   = '#f2f2f2'   # 눈(흰색)
PUPIL_COLOR = '#1f1f1f'   # 눈동자(검정)


def _oval_eye(cx, cy, w, h, dx=0.0, dy=0.0):
    # 둥근 사각형(흰 눈) + 눈동자(타원) 반환
    r = min(w, h) * 0.22
    return [
        FancyBboxPatch((cx - w/2.0, cy - h/2.0), w, h,
                       boxstyle='round,pad=0,rounding_size=' + str(r),
                       facecolor=EYE_COLOR, edgecolor='none'),
        Ellipse((cx + dx, cy + dy), w * 0.40, h * 0.55,
                facecolor=PUPIL_COLOR, edgecolor='none')
    ]


def _closed_eye(cx, cy, w, thickness=0.30):
    # 감긴(가는) 눈 반환
    r = thickness * 0.5
    return [
        FancyBboxPatch((cx - w/2.0, cy - thickness/2.0), w, thickness,
                       boxstyle='round,pad=0,rounding_size=' + str(r),
                       facecolor=EYE_COLOR, edgecolor='none')
    ]


def _arc_eye(cx, cy, w, h, theta1, theta2, lw=12):
    # 아치형 눈 반환 (0~180 = 위로 볼록, 180~360 = 아래로 볼록)
    return [Arc((cx, cy), w, h, theta1=theta1, theta2=theta2,
                linewidth=lw, edgecolor=EYE_COLOR)]


def _heart_eye(cx, cy, size=1.0):
    # 하트 모양 눈 반환
    t = np.linspace(0.0, 2.0*math.pi, 100)
    x = 16.0 * np.sin(t) ** 3
    y = 13.0 * np.cos(t) - 5.0*np.cos(2.0*t) - 2.0*np.cos(3.0*t) - np.cos(4.0*t)
    x = cx + x / 16.0 * size
    y = cy + y / 17.0 * size
    return [Polygon(np.column_stack([x, y]), closed=True,
                    facecolor=EYE_COLOR, edgecolor='none')]


def _eyebrow(cx, cy, angle_deg, length=1.0, thickness=0.12):
    # 중심점과 기울기로 두꺼운 눈썹(사각형) 반환
    a = math.radians(angle_deg)
    dx, dy = length/2.0*math.cos(a), length/2.0*math.sin(a)
    px, py = thickness/2.0*math.cos(a + math.pi/2.0), thickness/2.0*math.sin(a + math.pi/2.0)
    poly = [(cx - dx + px, cy - dy + py), (cx + dx + px, cy + dy + py),
            (cx + dx - px, cy + dy - py), (cx - dx - px, cy - dy - py)]
    return [Polygon(poly, closed=True, facecolor=EYE_COLOR, edgecolor='none')]

In [ ]:
class VectorFace:
    # Vector 스타일 얼굴의 상태(감정/시선/깜빡임)를 관리하는 클래스
    def __init__(self, fig, ax, fps=30):
        self.fig = fig
        self.ax = ax
        self.fps = fps
        self.frame = 0

        self.expr = 'normal'          # 현재 표정
        self.expr_left = 0.0          # 표정 유지 시간
        self._pick_expr(duration=(4.0, 7.0))

        self.blink = 0.0              # 0(뜸) ~ 1(감김)
        self.blink_hold = 0.0
        self.blink_left = random.uniform(2.0, 4.0)

        self.look = (0.0, 0.0)        # 시선 (x, y)
        self.look_left = random.uniform(0.8, 2.0)

    def _pick_expr(self, duration=None):
        exprs = ['normal', 'happy', 'sad', 'angry', 'surprised', 'sleepy', 'love']
        self.expr = random.choice(exprs)
        if duration is None:
            duration = (3.0, 6.0) if self.expr != 'normal' else (5.0, 9.0)
        self.expr_left = random.uniform(*duration)

    def step(self, dt):
        # 매 프레임마다 상태 갱신
        self.frame += 1

        # 표정 전환
        self.expr_left -= dt
        if self.expr_left <= 0.0:
            self._pick_expr()

        # 기본 표정일 때만 시선 이동
        if self.expr == 'normal':
            self.look_left -= dt
            if self.look_left <= 0.0:
                self.look = (random.uniform(-1.0, 1.0), random.uniform(-0.7, 0.7))
                self.look_left = random.uniform(0.8, 2.0)

        # 눈 깜빡임 (기본 표정에서 주기적으로 발생)
        self.blink_left -= dt
        if self.blink_left <= 0.0:
            self.blink = 1.0
            self.blink_hold = 0.15
            self.blink_left = random.uniform(2.0, 4.5)
        if self.blink > 0.0:
            self.blink_hold -= dt
            if self.blink_hold <= 0.0:
                self.blink = 0.0

    def build(self):
        # 현재 상태에 맞는 도형(패치) 목록 생성
        patches = []
        w, h = 1.7, 2.3          # 기본 눈 폭/높이
        lx, rx, cy = 3.6, 6.4, 3.3   # 왼/오른쪽 눈 중심 x, 눈 중심 y
        look_x, look_y = self.look
        blink = self.blink
        expr = self.expr

        if expr == 'happy':
            # 위로 볼록한 아치 눈 + 미소
            for cx in (lx, rx):
                patches += _arc_eye(cx, cy - 0.05, w, h*0.9, 0.0, 180.0)
            patches += _eyebrow(lx, 4.45, 0.0, 1.0, 0.12)
            patches += _eyebrow(rx, 4.45, 0.0, 1.0, 0.12)
            patches.append(Arc((5.0, 2.25), 1.15, 0.6,
                               theta1=180.0, theta2=360.0,
                               linewidth=10, edgecolor=EYE_COLOR))
        elif expr == 'sad':
            # 아래로 볼록한 아치 눈 + 안쪽으로 올라간 눈썹
            for cx in (lx, rx):
                patches += _arc_eye(cx, cy, w, h*0.75, 180.0, 360.0)
            patches += _eyebrow(lx - 0.35, 4.6, -20.0, 1.0, 0.12)
            patches += _eyebrow(rx + 0.35, 4.6, 20.0, 1.0, 0.12)
        elif expr == 'angry':
            # 가늘게 뜬 눈 + 안쪽으로 내려간 눈썹
            for cx in (lx, rx):
                patches += _closed_eye(cx, cy, w*0.85, 0.34)
            patches += _eyebrow(lx - 0.35, 4.5, 35.0, 1.15, 0.16)
            patches += _eyebrow(rx + 0.35, 4.5, -35.0, 1.15, 0.16)
        elif expr == 'surprised':
            # 크게 뜬 눈
            for cx in (lx, rx):
                patches += _oval_eye(cx, cy + 0.05, w*1.25, h*1.25)
        elif expr == 'sleepy':
            # 반쯤 감긴 눈 (Zzz는 render()에서 그림)
            for cx in (lx, rx):
                patches += _closed_eye(cx, cy - 0.1, w*0.95, 0.55)
            patches += _eyebrow(lx - 0.2, 4.15, 12.0, 1.0, 0.10)
            patches += _eyebrow(rx + 0.2, 4.15, -12.0, 1.0, 0.10)
        elif expr == 'love':
            # 하트 눈
            for cx in (lx, rx):
                patches += _heart_eye(cx, cy + 0.1, size=1.05)
        else:  # normal
            # 깜빡임(blink)에 따라 눈 높이 축소, 시선(look)에 따라 눈동자 이동
            hh = h * (1.0 - 0.9 * blink)
            for cx in (lx, rx):
                patches += _oval_eye(cx, cy, w, hh,
                                     dx=look_x * 0.35, dy=look_y * 0.45)
        return patches

    def render(self):
        # 현재 상태를 화면(축)에 그림
        self.ax.clear()
        for p in self.build():
            self.ax.add_patch(p)

        # 졸릴 때 떠다니는 Zzz
        if self.expr == 'sleepy':
            zy = 4.5 + 0.7 * math.sin(self.frame / 14.0)
            self.ax.text(7.5, zy, 'Z', color=EYE_COLOR, fontsize=40,
                         ha='center', va='center', fontweight='bold')

        self.ax.set_xlim(0.0, 10.0)
        self.ax.set_ylim(0.0, 6.0)
        self.ax.set_aspect('equal')
        self.ax.axis('off')

#### 3) 표정 미리보기

각 표정을 한 프레임씩 PNG로 저장해 노트북 안에서 확인한다. (백엔드가 TkAgg라도 이미지 출력은 자동으로 인라인 처리된다)

In [ ]:
from IPython.display import Image, display
import io

def preview_expr(expr, look=(0.0, 0.0), blink=0.0):
    fig = plt.figure(figsize=(10, 6), facecolor=BG_COLOR)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_facecolor(BG_COLOR)
    face = VectorFace(fig, ax)
    face.expr = expr
    face.look = look
    face.blink = blink
    face.render()
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=80)
    plt.close(fig)
    buf.seek(0)
    return Image(data=buf.getvalue())


for expr in ['normal', 'happy', 'sad', 'angry', 'surprised', 'sleepy', 'love']:
    print(expr)
    display(preview_expr(expr))

#### 4) 7인치 화면에 표정 애니메이션 실행

- 검은 배경의 전체 화면(또는 10:6 창)을 열고 감정/시선/깜빡임이 자동으로 바뀌는 얼굴을 실시간으로 그린다.
- `FULLSCREEN = True`이면 7인치 화면 전체를 덮는다. 화면 해상도에 맞지 않으면 `False`로 바꾼다.
- **중지: Kernel → Interrupt** (셀 실행 중 중단). 종료 후 5)번 정리 셀을 실행한다.

In [ ]:
# 설정
FULLSCREEN = True     # True: 전체 화면, False: 창 모드
FPS        = 30       # 프레임 레이트 (Jetson에서는 30이 무난)

# 7인치 화면(1024x600, 10:6) 비율에 맞춘 검은 그림 창 생성
fig = plt.figure(figsize=(10, 6), facecolor=BG_COLOR)
ax = fig.add_axes([0, 0, 1, 1])
ax.set_facecolor(BG_COLOR)

# 전체 화면 전환 (TkAgg)
if FULLSCREEN:
    try:
        plt.get_current_fig_manager().window.attributes('-fullscreen', True)
    except Exception:
        print("전체 화면 전환 실패. 창 모드로 실행합니다.")

face = VectorFace(fig, ax, fps=FPS)
print("표정 애니메이션 시작! 중지하려면 Kernel -> Interrupt")

# 실시간 애니메이션 루프 (KeyboardInterrupt로 중지)
try:
    while True:
        face.step(1.0 / FPS)
        face.render()
        fig.canvas.draw_idle()
        fig.canvas.flush_events()
        plt.pause(1.0 / FPS)
except KeyboardInterrupt:
    print("애니메이션 중지됨.")

#### 5) 실행 중지/정리

애니메이션이 남긴 그림 창을 닫는다.

In [ ]:
plt.close('all')
print("창이 닫혔습니다.")